In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (Unix seconds)
for df, ts_col in [(metric_df, "timestamp"), (trace_df, "timestamp"), (log_df, "timestamp"), (error_df, "timestamp")]:
    df[ts_col] = pd.to_datetime(df[ts_col], unit="s", utc=True)

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-20 05:54:38", utc=True)
window_end = pd.to_datetime("2024-01-20 06:24:38", utc=True)

# 1) Total row counts and unique cmdb_id lists for each file
files_summary = pd.DataFrame([
    {
        "file": "metric.csv",
        "total_rows": len(metric_df),
        "unique_cmdb_ids": ", ".join(sorted(metric_df["cmdb_id"].dropna().unique()))
    },
    {
        "file": "trace.csv",
        "total_rows": len(trace_df),
        "unique_cmdb_ids": ", ".join(sorted(trace_df["cmdb_id"].dropna().unique()))
    },
    {
        "file": "log.csv",
        "total_rows": len(log_df),
        "unique_cmdb_ids": ", ".join(sorted(log_df["cmdb_id"].dropna().unique()))
    },
    {
        "file": "error_logs.csv",
        "total_rows": len(error_df),
        "unique_cmdb_ids": ", ".join(sorted(error_df["cmdb_id"].dropna().unique()))
    }
])

# Helper to compute grouped percentiles and in-window counts
def group_percentiles_with_window_counts(df, group_cols, value_col="value"):
    # Global percentiles computed on full series per group (rule 9)
    agg = df.groupby(group_cols)[value_col].agg(
        count="count",
        p50=lambda x: x.quantile(0.5),
        p90=lambda x: x.quantile(0.9),
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99),
    ).reset_index()
    # Compute in-window counts (must filter by timestamp)
    in_window = df[(df["timestamp"] >= window_start) & (df["timestamp"] <= window_end)]
    in_window_counts = in_window.groupby(group_cols)[value_col].count().reset_index().rename(columns={value_col: "in_window_count"})
    # Merge, fill missing in_window_count with 0
    merged = agg.merge(in_window_counts, on=group_cols, how="left").fillna({"in_window_count": 0})
    # Ensure types and rounding for compactness
    for col in ["p50", "p90", "p95", "p99"]:
        merged[col] = merged[col].astype(float).round(6)
    merged["count"] = merged["count"].astype(int)
    merged["in_window_count"] = merged["in_window_count"].astype(int)
    # Sort by count desc and return
    merged = merged.sort_values("count", ascending=False).reset_index(drop=True)
    return merged

# 2) metric.csv grouping by (cmdb_id, kpi_name)
metric_grouped = group_percentiles_with_window_counts(metric_df, ["cmdb_id", "kpi_name"], value_col="value")
metric_top20 = metric_grouped[["cmdb_id", "kpi_name", "count", "p50", "p90", "p95", "p99", "in_window_count"]].head(20)

# 3) trace.csv grouping by (cmdb_id, trace_name)
trace_grouped = group_percentiles_with_window_counts(trace_df, ["cmdb_id", "trace_name"], value_col="value")
trace_top20 = trace_grouped[["cmdb_id", "trace_name", "count", "p50", "p90", "p95", "p99", "in_window_count"]].head(20)

# 4) log.csv grouping by (cmdb_id, log_name)
log_grouped = group_percentiles_with_window_counts(log_df, ["cmdb_id", "log_name"], value_col="value")
log_top20 = log_grouped[["cmdb_id", "log_name", "count", "p50", "p90", "p95", "p99", "in_window_count"]].head(20)

# 5) error_logs.csv: rows within incident window (up to top 20), sorted ascending by timestamp
error_in_window = error_df[(error_df["timestamp"] >= window_start) & (error_df["timestamp"] <= window_end)].copy()
error_in_window = error_in_window.sort_values("timestamp", ascending=True)
error_in_window_head = error_in_window[["timestamp", "cmdb_id", "message"]].head(20)

# Final compact outputs (variables to display)
files_summary, metric_top20, trace_top20, log_top20, len(error_df), error_in_window_head

```
Out[1]:
```
# Plain English summary of the execution results
summary = (
    "Summary of telemetry within the incident window (2024-01-20 05:54:38 to 2024-01-20 06:24:38 UTC):\n"
    "- metric.csv: 1850 rows. Observed cmdb_ids include adservice, cartservice, checkoutservice, currencyservice, "
    "emailservice, frontend, paymentservice, recommendationservice, shippingservice, redis, etc.\n"
    "- trace.csv: 3800 rows. Many trace entries; top trace groups are for checkoutservice.\n"
    "- log.csv: 460 rows. Same services present; redis has only 5 rows.\n"
    "- error_logs.csv: 0 rows (no error log lines in the dataset or within the window).\n\n"
    "- Grouped KPI findings (top groups shown in the previous output):\n"
    "  • Most top metric/trace/log groups show count = 25 and in_window_count = 25, meaning those series have 25 samples "
    "and all 25 samples fall inside the incident window (i.e., telemetry coverage for the window).\n"
    "  • One exception: redis log groups had count = 5 and in_window_count = 5 (smaller sample set).\n\n"
    "- Notable signal: checkoutservice shows relatively high latency metrics and trace durations (p50/p90/p95/p99 "
    "for checkout latency and related trace p95s are elevated in the top groups). This points to checkoutservice "
    "as the component with visible latency degradation in the provided telemetry.\n\n"
    "- No error log messages were present in error_logs.csv during the window to corroborate failures.\n\n"
    "Conclusion: telemetry is available for the listed services during the incident window; checkoutservice stands out "
    "with elevated latency in both metrics and traces. No error log entries were present to provide additional context."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(             file  total_rows                                    unique_cmdb_ids
0      metric.csv        1850  adservice, cartservice, checkoutservice, curre...
1       trace.csv        3800  checkoutservice, currencyservice, emailservice...
2         log.csv         460  adservice, cartservice, checkoutservice, curre...
3  error_logs.csv           0                                                   ,             cmdb_id    kpi_name  count           p50           p90           p95           p99  in_window_count
0         adservice         cpu     25  7.074800e-01  7.620050e-01  7.774970e-01  8.162990e-01               25
1         adservice      diskio     25  6.085527e+04  7.520401e+04  8.052342e+04  1.040495e+05               25
2         adservice  latency-50     25  3.009000e-03  3.023000e-03  3.023000e-03  3.030000e-03               25
3         adservice  latency-90     25  4.627000e-03  4.644000e-03  4.649000e-03  4.661000e-03               25
4         adservice         mem     25  1.080504e+08  1.081686e+08  1.081869e+08  1.081917e+08               25
5         adservice      socket     25  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00               25
6         adservice    workload     25  1.498025e+01  1.547422e+01  1.570551e+01  1.576637e+01               25
7       cartservice         cpu     25  2.251414e+00  2.658004e+00  2.785575e+00  2.852221e+00               25
8       cartservice      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               25
9       cartservice  latency-50     25  3.147000e-03  3.189000e-03  3.192000e-03  3.244000e-03               25
10      cartservice  latency-90     25  4.864000e-03  5.111000e-03  5.128000e-03  5.715000e-03               25
11      cartservice         mem     25  4.446522e+07  4.452528e+07  4.452927e+07  4.455186e+07               25
12      cartservice      socket     25  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00               25
13      cartservice    workload     25  2.509325e+01  2.630686e+01  2.659939e+01  2.706908e+01               25
14  checkoutservice         cpu     25  4.197390e-01  4.980640e-01  5.170550e-01  5.719500e-01               25
15  checkoutservice  latency-50     25  1.079730e-01  1.366290e-01  1.551780e-01  1.685290e-01               25
16  checkoutservice  latency-90     25  2.258870e-01  2.757440e-01  4.135630e-01  5.096430e-01               25
17  checkoutservice         mem     25  1.138865e+07  1.155858e+07  1.160667e+07  1.170097e+07               25
18  checkoutservice      socket     25  9.000000e+00  9.000000e+00  9.000000e+00  9.000000e+00               25
19  checkoutservice    workload     25  9.378000e-01  1.134243e+00  1.183325e+00  1.232413e+00               25,             cmdb_id                                trace_name  count         p50         p90         p95         p99  in_window_count
0   checkoutservice  trace.from_checkoutservice.duration_mean     25    0.003823    0.009179    0.010907    0.015062               25
1   checkoutservice   trace.from_checkoutservice.duration_p95     25    0.006024    0.043755    0.070361    0.076223               25
2   checkoutservice     trace.from_checkoutservice.error_rate     25    0.000000    0.000000    0.000000    0.000000               25
3   checkoutservice      trace.from_checkoutservice.row_count     25  240.000000  271.600000  284.800000  311.560000               25
4   checkoutservice  trace.from_frontendservice.duration_mean     25    0.100066    0.145160    0.148899    0.185463               25
5   checkoutservice   trace.from_frontendservice.duration_p95     25    0.202671    0.252851    0.291568    0.799365               25
6   checkoutservice     trace.from_frontendservice.error_rate     25    0.000000    0.000000    0.000000    0.000000               25
7   checkoutservice      trace.from_frontendservice.row_count     25   29.000000   33.000000   33.000000   34.520000               25
8   checkoutservice             trace.from_root.duration_mean     25    0.014349    0.017019    0.017929    0.018544               25
9   checkoutservice              trace.from_root.duration_p95     25    0.066811    0.073070    0.073964    0.074190               25
10  checkoutservice                trace.from_root.error_rate     25    0.000000    0.000000    0.000000    0.000000               25
11  checkoutservice                 trace.from_root.row_count     25  119.000000  136.600000  145.000000  159.920000               25
12  checkoutservice    trace.to_checkoutservice.duration_mean     25    0.003823    0.009179    0.010907    0.015062               25
13  checkoutservice     trace.to_checkoutservice.duration_p95     25    0.006024    0.043755    0.070361    0.076223               25
14  checkoutservice       trace.to_checkoutservice.error_rate     25    0.000000    0.000000    0.000000    0.000000               25
15  checkoutservice        trace.to_checkoutservice.row_count     25  240.000000  271.600000  284.800000  311.560000               25
16  checkoutservice    trace.to_currencyservice.duration_mean     25    0.000905    0.002002    0.002182    0.002268               25
17  checkoutservice     trace.to_currencyservice.duration_p95     25    0.000230    0.000313    0.000664    0.000954               25
18  checkoutservice       trace.to_currencyservice.error_rate     25    0.000000    0.000000    0.000000    0.000000               25
19  checkoutservice        trace.to_currencyservice.row_count     25   95.000000  112.600000  121.000000  135.920000               25,                   cmdb_id         log_name  count     p50     p90     p95      p99  in_window_count
0               adservice  log.error_count     25     0.0     0.0     0.0     0.00               25
1               adservice    log.row_count     25   448.0   462.8   464.8   470.32               25
2             cartservice  log.error_count     25     0.0     0.0     0.0     0.00               25
3             cartservice    log.row_count     25   764.0   797.8   807.4   815.84               25
4         checkoutservice  log.error_count     25     0.0     0.0     0.0     0.00               25
5         checkoutservice    log.row_count     25    87.0    99.0    99.0   105.08               25
6         currencyservice  log.error_count     25     0.0     0.0     0.0     0.00               25
7         currencyservice    log.row_count     25  2323.0  2448.8  2482.8  2498.12               25
8            emailservice  log.error_count     25     0.0     0.0     0.0     0.00               25
9            emailservice    log.row_count     25    29.0    33.0    33.0    35.28               25
10               frontend  log.error_count     25     0.0     0.0     0.0     0.00               25
11               frontend    log.row_count     25  2644.0  2738.6  2749.8  2799.40               25
12         paymentservice  log.error_count     25     0.0     0.0     0.0     0.00               25
13         paymentservice    log.row_count     25    58.0    66.0    66.0    70.56               25
14  recommendationservice  log.error_count     25     0.0     0.0     0.0     0.00               25
15  recommendationservice    log.row_count     25   555.0   585.0   587.4   588.76               25
16        shippingservice    log.row_count     25   446.0   473.6   482.4   491.60               25
17        shippingservice  log.error_count     25     0.0     0.0     0.0     0.00               25
18                  redis    log.row_count      5     5.0     5.0     5.0     5.00                5
19                  redis  log.error_count      5     0.0     0.0     0.0     0.00                5, 0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd

# Reuse existing metric_df and window_start/window_end from the session
# If they are not present (e.g., fresh kernel), reload metric.csv and parse timestamps
try:
    metric_df
    window_start
    window_end
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df["timestamp"] = pd.to_datetime(metric_df["timestamp"], unit="s", utc=True)
    window_start = pd.to_datetime("2024-01-20 05:54:38", utc=True)
    window_end = pd.to_datetime("2024-01-20 06:24:38", utc=True)

# Target cmdb_ids
targets = ["checkoutservice", "currencyservice", "emailservice", "productcatalogservice", "recommendationservice"]

# Restrict to target components (for analysis groups). Note: percentiles must be computed on full series per group
metric_filtered = metric_df[metric_df["cmdb_id"].isin(targets)].copy()

# Compute global p5, p95 and total count per (cmdb_id, kpi_name)
agg = metric_filtered.groupby(["cmdb_id", "kpi_name"])["value"].agg(
    count="count",
    p5=lambda x: x.quantile(0.05),
    p95=lambda x: x.quantile(0.95)
).reset_index()

# Compute in-window rows for the same target set
in_window = metric_filtered[(metric_filtered["timestamp"] >= window_start) & (metric_filtered["timestamp"] <= window_end)].copy()

# in_window counts per group
in_window_counts = in_window.groupby(["cmdb_id", "kpi_name"])["value"].count().reset_index().rename(columns={"value": "in_window_count"})
agg = agg.merge(in_window_counts, on=["cmdb_id", "kpi_name"], how="left").fillna({"in_window_count": 0})
agg["in_window_count"] = agg["in_window_count"].astype(int)

# For each group, compute anomalies within the in-window rows
results = []
for _, row in agg.iterrows():
    cmdb = row["cmdb_id"]
    kpi = row["kpi_name"]
    count = int(row["count"])
    p5 = float(row["p5"])
    p95 = float(row["p95"])
    in_count = int(row["in_window_count"])
    group_window_rows = in_window[(in_window["cmdb_id"] == cmdb) & (in_window["kpi_name"] == kpi)]
    # High anomalies: value >= p95
    if not group_window_rows.empty and not np.isnan(p95):
        high_rows = group_window_rows[group_window_rows["value"] >= p95]
    else:
        high_rows = group_window_rows.iloc[0:0]  # empty
    anomalies_high = int(high_rows.shape[0])
    if anomalies_high > 0:
        earliest_high_ts = high_rows["timestamp"].min().isoformat()
        max_value_in_window = float(high_rows["value"].max())
        high_severity = np.nan if p95 == 0 else (max_value_in_window - p95) / p95
    else:
        earliest_high_ts = pd.NA
        max_value_in_window = np.nan
        high_severity = np.nan
    # Low anomalies: value <= p5
    if not group_window_rows.empty and not np.isnan(p5):
        low_rows = group_window_rows[group_window_rows["value"] <= p5]
    else:
        low_rows = group_window_rows.iloc[0:0]
    anomalies_low = int(low_rows.shape[0])
    if anomalies_low > 0:
        earliest_low_ts = low_rows["timestamp"].min().isoformat()
        min_value_in_window = float(low_rows["value"].min())
        low_severity = np.nan if p5 == 0 else (p5 - min_value_in_window) / p5
    else:
        earliest_low_ts = pd.NA
        min_value_in_window = np.nan
        low_severity = np.nan

    # Only keep groups that have any anomalies
    if anomalies_high > 0 or anomalies_low > 0:
        results.append({
            "cmdb_id": cmdb,
            "kpi_name": kpi,
            "count": count,
            "p5": round(p5, 6),
            "p95": round(p95, 6),
            "in_window_count": in_count,
            "anomalies_high": anomalies_high,
            "earliest_high_timestamp": earliest_high_ts,
            "max_value_in_window": round(max_value_in_window, 6) if not np.isnan(max_value_in_window) else np.nan,
            "high_severity": round(high_severity, 6) if not np.isnan(high_severity) else np.nan,
            "anomalies_low": anomalies_low,
            "earliest_low_timestamp": earliest_low_ts,
            "min_value_in_window": round(min_value_in_window, 6) if not np.isnan(min_value_in_window) else np.nan,
            "low_severity": round(low_severity, 6) if not np.isnan(low_severity) else np.nan
        })

result_df = pd.DataFrame(results)

# Sort by (anomalies_high + anomalies_low) desc, then in_window_count desc. Limit to top 20.
if not result_df.empty:
    result_df["anomaly_total"] = result_df["anomalies_high"] + result_df["anomalies_low"]
    result_df = result_df.sort_values(["anomaly_total", "in_window_count"], ascending=[False, False]).reset_index(drop=True)
    result_df = result_df.drop(columns=["anomaly_total"])
else:
    # keep empty with specified columns
    cols = ["cmdb_id","kpi_name","count","p5","p95","in_window_count","anomalies_high","earliest_high_timestamp",
            "max_value_in_window","high_severity","anomalies_low","earliest_low_timestamp","min_value_in_window","low_severity"]
    result_df = pd.DataFrame(columns=cols)

# Prepare a small summary count
summary = pd.DataFrame([{
    "groups_analyzed": int(agg.shape[0]),
    "groups_with_anomalies": int(result_df.shape[0])
}])

# Final compact outputs: top 20 anomaly rows and the summary
result_df_head = result_df.head(20)
result_df_head, summary

```
Out[2]:
```
Summary (plain English)

- Scope: I analyzed metric.csv for 5 services: checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice. I examined 34 (cmdb_id, kpi_name) groups in total. All 34 groups showed at least one anomaly inside the incident window (2024-01-20 05:54:38 to 2024-01-20 06:24:38 UTC).

Key findings (compact, prioritized by anomaly counts and severity):
- Checkoutservice shows the strongest and most meaningful anomalies:
  - latency-90: p5=0.21169, p95=0.41356; 2 high anomalies in-window, earliest high at 2024-01-20T05:59:00Z, max in-window = 0.529208, high_severity ≈ 0.2796 (largest severity observed).
  - cpu: p5=0.337822, p95=0.517055; 2 high anomalies (earliest 2024-01-20T06:02:00Z), max = 0.588059, high_severity ≈ 0.1373.
  - latency-50: p5=0.081447, p95=0.155178; 2 high anomalies (earliest 2024-01-20T06:10:00Z), max = 0.171452, high_severity ≈ 0.1049.
  - mem, workload also show 2 highs and 2 lows with modest severities (workload high_severity ≈ 0.0516).
  - Earliest checkoutservice anomalous samples appear around 2024-01-20T05:59:00Z–06:02:00Z.

- Currencyservice notable anomalies:
  - socket: p5=3.0, p95=3.0; 24 high anomalies (earliest high 2024-01-20T05:57:00Z), max = 3.466667, high_severity ≈ 0.1556.
  - latency-90: p5=0.028595, p95=0.046152; 2 high anomalies (earliest 2024-01-20T06:02:00Z), max = 0.051874, high_severity ≈ 0.1240.
  - cpu and mem show 2 highs and 2 lows with small severities (cpu high_severity ≈ 0.0082).
  - Many currencyservice groups have earliest anomalies at 2024-01-20T05:57:00Z–06:10:00Z.

- Emailservice:
  - latency-50 and latency-90: each had 2 high anomalies (earliest highs ~2024-01-20T06:11:00Z) with moderate severity (latency-50 high_severity ≈ 0.0662; latency-90 high_severity ≈ 0.0381).
  - socket and diskio show many anomalies; diskio has p5=p95=0 so severity is NaN (division by zero), meaning the percentile baseline is zero and all nonzero behavior is flagged but severity undefined.

- Productcatalogservice and recommendationservice:
  - diskio fields: p5=p95=0 and show anomalies in-window (severity NaN).
  - socket fields have equal p5/p95 and show both high and low counts but zero severity (equal values).

Additional notes:
- Many groups have in_window_count = 25 (full coverage across the window); a few groups had different in-window counts but most anomalies are fully inside the incident window.
- Several diskio groups have p95==0 (so severity cannot be computed—reported as NaN). Multiple socket groups have identical p5 and p95 (constant baseline), producing high and low counts of identical values with zero computed severity.

Bottom line:
- Anomalies are widespread across the five services during the incident window. The most actionable signals (non-NaN, larger severity) point to checkoutservice (especially latency-90, latency-50, CPU) and currencyservice (socket, latency-90) as the most impacted components/KPIs in this dataset.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  count            p5           p95  in_window_count  anomalies_high    earliest_high_timestamp  max_value_in_window  high_severity  anomalies_low     earliest_low_timestamp  min_value_in_window  low_severity
0         checkoutservice      socket     25  9.000000e+00  9.000000e+00               25              25  2024-01-20T05:57:00+00:00         9.000000e+00       0.000000             25  2024-01-20T05:57:00+00:00         9.000000e+00      0.000000
1         currencyservice      diskio     25  0.000000e+00  0.000000e+00               25              25  2024-01-20T05:57:00+00:00         0.000000e+00            NaN             25  2024-01-20T05:57:00+00:00         0.000000e+00           NaN
2            emailservice      diskio     25  0.000000e+00  0.000000e+00               25              25  2024-01-20T05:57:00+00:00         0.000000e+00            NaN             25  2024-01-20T05:57:00+00:00         0.000000e+00           NaN
3   productcatalogservice      diskio     25  0.000000e+00  0.000000e+00               25              25  2024-01-20T05:57:00+00:00         0.000000e+00            NaN             25  2024-01-20T05:57:00+00:00         0.000000e+00           NaN
4   productcatalogservice      socket     25  3.000000e+00  3.000000e+00               25              25  2024-01-20T05:57:00+00:00         3.000000e+00       0.000000             25  2024-01-20T05:57:00+00:00         3.000000e+00      0.000000
5   recommendationservice      diskio     25  0.000000e+00  0.000000e+00               25              25  2024-01-20T05:57:00+00:00         0.000000e+00            NaN             25  2024-01-20T05:57:00+00:00         0.000000e+00           NaN
6   recommendationservice      socket     25  4.000000e+00  4.000000e+00               25              25  2024-01-20T05:57:00+00:00         4.000000e+00       0.000000             25  2024-01-20T05:57:00+00:00         4.000000e+00      0.000000
7         currencyservice      socket     25  3.000000e+00  3.000000e+00               25              25  2024-01-20T05:57:00+00:00         3.466667e+00       0.155556             24  2024-01-20T05:57:00+00:00         3.000000e+00      0.000000
8            emailservice      socket     25  3.000000e+00  1.006333e+01               25               2  2024-01-20T06:10:00+00:00         1.038333e+01       0.031799             12  2024-01-20T05:57:00+00:00         3.000000e+00      0.000000
9            emailservice  latency-50     25  3.000000e-03  5.739800e-02               25               2  2024-01-20T06:11:00+00:00         6.119600e-02       0.066184             11  2024-01-20T05:59:00+00:00         3.000000e-03      0.000000
10           emailservice  latency-90     25  4.600000e-03  9.124700e-02               25               2  2024-01-20T06:11:00+00:00         9.472700e-02       0.038138             11  2024-01-20T05:59:00+00:00         4.600000e-03      0.000000
11        checkoutservice         cpu     25  3.378220e-01  5.170550e-01               25               2  2024-01-20T06:02:00+00:00         5.880590e-01       0.137323              2  2024-01-20T06:11:00+00:00         3.094070e-01      0.084114
12        checkoutservice  latency-50     25  8.144700e-02  1.551780e-01               25               2  2024-01-20T06:10:00+00:00         1.714520e-01       0.104870              2  2024-01-20T06:06:00+00:00         7.735700e-02      0.050208
13        checkoutservice  latency-90     25  2.116900e-01  4.135630e-01               25               2  2024-01-20T05:59:00+00:00         5.292080e-01       0.279631              2  2024-01-20T06:06:00+00:00         2.046590e-01      0.033216
14        checkoutservice         mem     25  1.128141e+07  1.160667e+07               25               2  2024-01-20T06:04:00+00:00         1.172862e+07       0.010507              2  2024-01-20T06:01:00+00:00         1.114754e+07      0.011867
15        checkoutservice    workload     25  7.657830e-01  1.183325e+00               25               2  2024-01-20T06:02:00+00:00         1.244433e+00       0.051641              2  2024-01-20T06:00:00+00:00         6.201000e-01      0.190241
16        currencyservice         cpu     25  2.834157e+01  3.309944e+01               25               2  2024-01-20T06:10:00+00:00         3.336944e+01       0.008157              2  2024-01-20T05:57:00+00:00         2.764416e+01      0.024607
17        currencyservice  latency-50     25  3.642000e-03  3.938000e-03               25               2  2024-01-20T05:57:00+00:00         4.036000e-03       0.025017              2  2024-01-20T06:06:00+00:00         3.629000e-03      0.003679
18        currencyservice  latency-90     25  2.859500e-02  4.615200e-02               25               2  2024-01-20T06:02:00+00:00         5.187400e-02       0.123984              2  2024-01-20T06:06:00+00:00         2.802300e-02      0.020001
19        currencyservice         mem     25  4.470452e+07  4.568467e+07               25               2  2024-01-20T05:58:00+00:00         4.579403e+07       0.002394              2  2024-01-20T06:08:00+00:00         4.447764e+07      0.005075,    groups_analyzed  groups_with_anomalies
0               34                     34)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse trace_df and window_start/window_end if present, otherwise load/parse
try:
    trace_df
    window_start
    window_end
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df["timestamp"] = pd.to_datetime(trace_df["timestamp"], unit="s", utc=True)
    window_start = pd.to_datetime("2024-01-20 05:54:38", utc=True)
    window_end = pd.to_datetime("2024-01-20 06:24:38", utc=True)

# Target cmdb_ids
targets = ["checkoutservice", "currencyservice", "emailservice", "productcatalogservice", "recommendationservice"]

# Restrict to target components for grouping (global percentiles computed on full series for these targets)
trace_filtered = trace_df[trace_df["cmdb_id"].isin(targets)].copy()

# 1) Compute global p5, p95 and count per (cmdb_id, trace_name)
agg = trace_filtered.groupby(["cmdb_id", "trace_name"])["value"].agg(
    count="count",
    p5=lambda x: x.quantile(0.05),
    p95=lambda x: x.quantile(0.95)
).reset_index()

# 2) Filter in-window rows
in_window = trace_filtered[(trace_filtered["timestamp"] >= window_start) & (trace_filtered["timestamp"] <= window_end)].copy()
in_window_counts = in_window.groupby(["cmdb_id", "trace_name"])["value"].count().reset_index().rename(columns={"value": "in_window_count"})
agg = agg.merge(in_window_counts, on=["cmdb_id", "trace_name"], how="left").fillna({"in_window_count": 0})
agg["in_window_count"] = agg["in_window_count"].astype(int)

# 3) For each group compute anomalies within in-window rows
results = []
for _, row in agg.iterrows():
    cmdb = row["cmdb_id"]
    trace_name = row["trace_name"]
    count = int(row["count"])
    p5 = float(row["p5"])
    p95 = float(row["p95"])
    in_count = int(row["in_window_count"])
    group_window_rows = in_window[(in_window["cmdb_id"] == cmdb) & (in_window["trace_name"] == trace_name)]
    # High anomalies
    if not group_window_rows.empty and not np.isnan(p95):
        high_rows = group_window_rows[group_window_rows["value"] >= p95]
    else:
        high_rows = group_window_rows.iloc[0:0]
    anomalies_high = int(high_rows.shape[0])
    if anomalies_high > 0:
        earliest_high_ts = high_rows["timestamp"].min().isoformat()
        max_value_in_window = float(high_rows["value"].max())
        high_severity = np.nan if p95 == 0 else (max_value_in_window - p95) / p95
    else:
        earliest_high_ts = pd.NA
        max_value_in_window = np.nan
        high_severity = np.nan
    # Low anomalies
    if not group_window_rows.empty and not np.isnan(p5):
        low_rows = group_window_rows[group_window_rows["value"] <= p5]
    else:
        low_rows = group_window_rows.iloc[0:0]
    anomalies_low = int(low_rows.shape[0])
    if anomalies_low > 0:
        earliest_low_ts = low_rows["timestamp"].min().isoformat()
        min_value_in_window = float(low_rows["value"].min())
        low_severity = np.nan if p5 == 0 else (p5 - min_value_in_window) / p5
    else:
        earliest_low_ts = pd.NA
        min_value_in_window = np.nan
        low_severity = np.nan

    # Keep only groups with any anomalies
    if anomalies_high > 0 or anomalies_low > 0:
        results.append({
            "cmdb_id": cmdb,
            "trace_name": trace_name,
            "count": count,
            "p5": round(p5, 6),
            "p95": round(p95, 6),
            "in_window_count": in_count,
            "anomalies_high": anomalies_high,
            "earliest_high_timestamp": earliest_high_ts,
            "max_value_in_window": round(max_value_in_window, 6) if not np.isnan(max_value_in_window) else np.nan,
            "high_severity": round(high_severity, 6) if not np.isnan(high_severity) else np.nan,
            "anomalies_low": anomalies_low,
            "earliest_low_timestamp": earliest_low_ts,
            "min_value_in_window": round(min_value_in_window, 6) if not np.isnan(min_value_in_window) else np.nan,
            "low_severity": round(low_severity, 6) if not np.isnan(low_severity) else np.nan
        })

result_df = pd.DataFrame(results)

# 4) Sort and limit to top 20 by anomaly count then in_window_count
if not result_df.empty:
    result_df["anomaly_total"] = result_df["anomalies_high"] + result_df["anomalies_low"]
    result_df = result_df.sort_values(["anomaly_total", "in_window_count"], ascending=[False, False]).reset_index(drop=True)
    result_df = result_df.drop(columns=["anomaly_total"])
else:
    cols = ["cmdb_id","trace_name","count","p5","p95","in_window_count","anomalies_high","earliest_high_timestamp",
            "max_value_in_window","high_severity","anomalies_low","earliest_low_timestamp","min_value_in_window","low_severity"]
    result_df = pd.DataFrame(columns=cols)

result_df_head = result_df.head(20)

# 5) Focused list: checkoutservice and trace_name contains 'to_' or 'from_'
mask_call_edges = (result_df["cmdb_id"] == "checkoutservice") & \
                  (result_df["trace_name"].str.contains("to_|from_", na=False))
focused = result_df[mask_call_edges].copy()

# Sort focused by earliest_high_timestamp ascending (put NA at end)
def _sort_key(ts):
    if pd.isna(ts):
        return pd.Timestamp.max.tz_localize("UTC")
    try:
        return pd.to_datetime(ts)
    except Exception:
        return pd.Timestamp.max.tz_localize("UTC")

if not focused.empty:
    focused["earliest_high_sort"] = focused["earliest_high_timestamp"].apply(_sort_key)
    focused = focused.sort_values("earliest_high_sort", ascending=True).drop(columns=["earliest_high_sort"]).reset_index(drop=True)
else:
    focused = focused

focused_head = focused.head(20)

# Final compact outputs
result_df_head, focused_head

```
Out[3]:
```
Summary (plain English)

Scope and overall note
- I analyzed trace.csv for the five services you specified during the incident window (2024-01-20 05:54:38 — 2024-01-20 06:24:38 UTC).
- Many trace "error_rate" series had p5 = p95 = 0.0; these show up as anomaly counts (because values equal the percentile) but their severity is undefined (NaN) and they are not actionable by severity alone.
- The earliest anomalous timestamps in multiple groups start at 2024-01-20T05:57:00Z.

Top trace anomalies (compact highlights)
- Lots of trace error_rate groups across services (checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice) have p5=p95=0 and show 25 in-window samples flagged — severity is NaN (baseline zero), so treat these as uninformative for magnitude.
- Checkoutservice call/edge and root traces show meaningful timing and count anomalies:
  - trace.from_frontendservice.duration_p95 — count 25, p95=0.291568, 2 high anomalies; earliest high 2024-01-20T05:59:00Z, max = 0.957029, high_severity ≈ 2.28 (very large spike above baseline).
  - trace.to_paymentservice.duration_mean and duration_p95 — each had 2 high anomalies with very large severities:
    - duration_mean: p95=0.000770, max in window = 0.005869, high_severity ≈ 6.63
    - duration_p95: p95=0.002816, max in window = 0.019770, high_severity ≈ 6.02
  - trace.to_checkoutservice.duration_mean and trace.from_checkoutservice.duration_mean — both show 2 high anomalies (earliest 2024-01-20T05:59:00Z) with max ≈ 0.016238 and high_severity ≈ 0.4887.
  - Row-count edges for checkoutservice show multiple highs:
    - trace.from_frontendservice.row_count — 5 highs, earliest high 2024-01-20T06:02:00Z, max 35, high_severity ≈ 0.0606.
    - trace.to_paymentservice.row_count — 4 highs, earliest high 2024-01-20T06:02:00Z, max 36.
    - trace.to_currencyservice.row_count and trace.to_productcatalogservice.row_count — each show 2 highs (earliest highs ~06:02:00Z), with max values and moderate severities (e.g., to_productcatalogservice.max=104, high_severity ≈ 0.1738).
- Timing pattern: many of the non-zero, high-severity trace anomalies for checkoutservice appear around 05:59:00Z and a concentrated spike of row_count/duration anomalies around ~06:02:00Z.

Focused checkoutservice call-edge summary (call-chain view)
- Call-edge traces (trace_name containing "to_" or "from_") for checkoutservice indicate:
  - Error rate traces are all zero (p5=p95=0) and flagged but not useful for severity.
  - Duration spikes affecting calls to paymentservice, checkoutservice, currencyservice and from frontend are evident — largest relative spikes seen for calls involving paymentservice (severity > 6).
  - Row-count anomalies show increased throughput for several edges at ~06:02:00Z (from_frontendservice.row_count, to_paymentservice.row_count, to_productcatalogservice.row_count, to_currencyservice.row_count).

Actionable takeaways
- Checkoutservice shows the most actionable trace anomalies: large duration spikes on edges (especially to/from paymentservice and from frontend) and elevated row counts around ~06:02Z — investigate checkoutservice call paths and paymentservice interactions.
- Many error_rate traces are zero and not informative; focus on duration and row_count anomalies for root-cause clues.
- The timing suggests an event beginning ~05:59:00Z with concentrated activity around ~06:02:00Z — inspect traces/logs and service interactions at those times (checkoutservice → paymentservice / productcatalogservice / currencyservice).

If you want, I can extract the exact focused table rows for checkoutservice call-edges (with the exact timestamps and numeric values shown) for copy-paste into an incident report.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                   trace_name  count   p5  p95  in_window_count  anomalies_high    earliest_high_timestamp  max_value_in_window  high_severity  anomalies_low     earliest_low_timestamp  min_value_in_window  low_severity
0         checkoutservice        trace.from_checkoutservice.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
1         checkoutservice        trace.from_frontendservice.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
2         checkoutservice                   trace.from_root.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
3         checkoutservice          trace.to_checkoutservice.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
4         checkoutservice          trace.to_currencyservice.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
5         checkoutservice             trace.to_emailservice.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
6         checkoutservice           trace.to_paymentservice.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
7         checkoutservice    trace.to_productcatalogservice.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
8         currencyservice        trace.from_checkoutservice.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
9         currencyservice        trace.from_frontendservice.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
10        currencyservice                   trace.from_root.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
11           emailservice        trace.from_checkoutservice.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
12           emailservice                   trace.from_root.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
13  productcatalogservice        trace.from_checkoutservice.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
14  productcatalogservice        trace.from_frontendservice.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
15  productcatalogservice  trace.from_recommendationservice.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
16  productcatalogservice                   trace.from_root.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
17  recommendationservice        trace.from_frontendservice.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
18  recommendationservice  trace.from_recommendationservice.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN
19  recommendationservice                   trace.from_root.error_rate     25  0.0  0.0               25              25  2024-01-20T05:57:00+00:00                  0.0            NaN             25  2024-01-20T05:57:00+00:00                  0.0           NaN,             cmdb_id                                 trace_name  count          p5         p95  in_window_count  anomalies_high    earliest_high_timestamp  max_value_in_window  high_severity  anomalies_low     earliest_low_timestamp  min_value_in_window  low_severity
0   checkoutservice      trace.from_checkoutservice.error_rate     25    0.000000    0.000000               25              25  2024-01-20T05:57:00+00:00             0.000000            NaN             25  2024-01-20T05:57:00+00:00             0.000000           NaN
1   checkoutservice      trace.from_frontendservice.error_rate     25    0.000000    0.000000               25              25  2024-01-20T05:57:00+00:00             0.000000            NaN             25  2024-01-20T05:57:00+00:00             0.000000           NaN
2   checkoutservice                 trace.from_root.error_rate     25    0.000000    0.000000               25              25  2024-01-20T05:57:00+00:00             0.000000            NaN             25  2024-01-20T05:57:00+00:00             0.000000           NaN
3   checkoutservice        trace.to_checkoutservice.error_rate     25    0.000000    0.000000               25              25  2024-01-20T05:57:00+00:00             0.000000            NaN             25  2024-01-20T05:57:00+00:00             0.000000           NaN
4   checkoutservice        trace.to_currencyservice.error_rate     25    0.000000    0.000000               25              25  2024-01-20T05:57:00+00:00             0.000000            NaN             25  2024-01-20T05:57:00+00:00             0.000000           NaN
5   checkoutservice           trace.to_emailservice.error_rate     25    0.000000    0.000000               25              25  2024-01-20T05:57:00+00:00             0.000000            NaN             25  2024-01-20T05:57:00+00:00             0.000000           NaN
6   checkoutservice         trace.to_paymentservice.error_rate     25    0.000000    0.000000               25              25  2024-01-20T05:57:00+00:00             0.000000            NaN             25  2024-01-20T05:57:00+00:00             0.000000           NaN
7   checkoutservice  trace.to_productcatalogservice.error_rate     25    0.000000    0.000000               25              25  2024-01-20T05:57:00+00:00             0.000000            NaN             25  2024-01-20T05:57:00+00:00             0.000000           NaN
8   checkoutservice     trace.to_checkoutservice.duration_mean     25    0.003671    0.010907               25               2  2024-01-20T05:59:00+00:00             0.016238       0.488712              2  2024-01-20T05:57:00+00:00             0.003634      0.009961
9   checkoutservice   trace.from_frontendservice.duration_mean     25    0.080185    0.148899               25               2  2024-01-20T05:59:00+00:00             0.196901       0.322385              2  2024-01-20T06:04:00+00:00             0.076987      0.039890
10  checkoutservice   trace.from_checkoutservice.duration_mean     25    0.003671    0.010907               25               2  2024-01-20T05:59:00+00:00             0.016238       0.488712              2  2024-01-20T05:57:00+00:00             0.003634      0.009961
11  checkoutservice    trace.from_frontendservice.duration_p95     25    0.119866    0.291568               25               2  2024-01-20T05:59:00+00:00             0.957029       2.282358              2  2024-01-20T06:04:00+00:00             0.118395      0.012277
12  checkoutservice       trace.to_paymentservice.duration_p95     25    0.000289    0.002816               25               2  2024-01-20T06:00:00+00:00             0.019770       6.020455              2  2024-01-20T05:57:00+00:00             0.000262      0.093458
13  checkoutservice      trace.to_paymentservice.duration_mean     25    0.000222    0.000770               25               2  2024-01-20T06:00:00+00:00             0.005869       6.625163              2  2024-01-20T05:57:00+00:00             0.000214      0.037180
14  checkoutservice     trace.to_currencyservice.duration_mean     25    0.000142    0.002182               25               2  2024-01-20T06:00:00+00:00             0.002282       0.045817              2  2024-01-20T06:20:00+00:00             0.000141      0.008712
15  checkoutservice         trace.to_checkoutservice.row_count     25  109.000000  284.800000               25               2  2024-01-20T06:02:00+00:00           319.000000       0.120084              2  2024-01-20T05:57:00+00:00            61.000000      0.440367
16  checkoutservice         trace.to_currencyservice.row_count     25   48.000000  121.000000               25               2  2024-01-20T06:02:00+00:00           140.000000       0.157025              2  2024-01-20T05:57:00+00:00            26.000000      0.458333
17  checkoutservice          trace.to_paymentservice.row_count     25   12.200000   33.000000               25               4  2024-01-20T06:02:00+00:00            36.000000       0.090909              2  2024-01-20T05:57:00+00:00             7.000000      0.426230
18  checkoutservice       trace.from_frontendservice.row_count     25   12.200000   33.000000               25               5  2024-01-20T06:02:00+00:00            35.000000       0.060606              2  2024-01-20T05:57:00+00:00             7.000000      0.426230
19  checkoutservice   trace.to_productcatalogservice.row_count     25   35.800000   88.600000               25               2  2024-01-20T06:02:00+00:00           104.000000       0.173815              2  2024-01-20T05:57:00+00:00            19.000000      0.469274)```
```